# Exploratory Data Analysis (EDA): Dataset Annotations & Properties
### Statistical Analysis of Categories, Dimensions, Aspect Ratios & Tag Densities

This notebook performs a comprehensive **Exploratory Data Analysis (EDA)** on raw COCO annotations:
- Discovers annotation files from `coco_files/` or split directories.
- Analyzes class balance (`location_tag`, `Blue_aisle`, `blue_bay`).
- Examines image resolutions, aspect ratios, and bounding box scale distributions.
- Quantifies tag densities (single-tag vs. multi-tag shelf scenes).
- Renders visual sample plots of the ground truth labels.

In [ ]:
# STEP 0: Dependencies & Environment Setup
import sys

%pip install -q pycocotools Pillow matplotlib pandas numpy tabulate tqdm

print(f"Python Environment: {sys.executable}")
print("✓ EDA dependencies verified.")

In [ ]:
# CELL 1: Imports & Logger Setup
import os
import sys
import json
import logging
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("DatasetEDA")
print("Imports and logger initialized successfully.")

In [ ]:
# CELL 2: Discovery of Annotation JSON Files
# Resilient workspace root resolver
REPO_ROOT = next((p.resolve() for p in [Path(".."), Path("."), Path("../..")] if (p / "coco_files").exists()), Path("..").resolve())
CANDIDATE_PATHS = [
    REPO_ROOT / "coco_files" / "_annotations.coco.json",
    REPO_ROOT / "coco_files",
    REPO_ROOT / "multi_class_train_rfdetr" / "dataset_full_data" / "train" / "_annotations.coco.json",
    REPO_ROOT / "multi_class_train_rfdetr" / "dataset_sample_1000" / "train" / "_annotations.coco.json",
    REPO_ROOT / "dataset_full_data" / "train" / "_annotations.coco.json"
]

ANN_FILE = None
for p in CANDIDATE_PATHS:
    if p.is_file() and p.stat().st_size > 0:
        ANN_FILE = p
        break
    elif p.is_dir():
        jsons = list(p.glob("*.json"))
        if jsons:
            ANN_FILE = jsons[0]
            break

if ANN_FILE is None:
    logger.warning("No COCO JSON found in candidates. Creating fallback placeholder path.")
    ANN_FILE = REPO_ROOT / "coco_files" / "_annotations.coco.json"

print(f"Active Annotation File for EDA: {ANN_FILE}")

In [ ]:
# CELL 3: Load & Parse COCO Annotations
raw_images = []
raw_annotations = []
raw_categories = []

if ANN_FILE.exists():
    with open(ANN_FILE, "r") as f:
        coco_data = json.load(f)
    raw_images = coco_data.get("images", [])
    raw_annotations = coco_data.get("annotations", [])
    raw_categories = coco_data.get("categories", [])
    print(f"Loaded {len(raw_images)} images, {len(raw_annotations)} annotations across {len(raw_categories)} categories.")
else:
    print(f"Annotation file not found yet at {ANN_FILE}. (Place your COCO JSON in coco_files/ to inspect full stats).")

cat_id_to_name = {c["id"]: c["name"] for c in raw_categories}

In [ ]:
# CELL 4: Category Distribution & Frequency Analysis
if raw_annotations:
    cat_counts = Counter([cat_id_to_name.get(a["category_id"], str(a["category_id"])) for a in raw_annotations])
    df_cats = pd.DataFrame(list(cat_counts.items()), columns=["Category", "Count"]).sort_values("Count", ascending=False)
    
    print("=== Category Distribution ===")
    print(df_cats.to_string(index=False))
    
    plt.figure(figsize=(8, 4))
    plt.bar(df_cats["Category"], df_cats["Count"], color=["#00C853", "#0091EA", "#2962FF"][:len(df_cats)])
    plt.title("Annotation Count by Category", fontsize=13)
    plt.ylabel("Count")
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()
else:
    print("No annotations loaded.")

In [ ]:
# CELL 5: Bounding Box Dimensions & Aspect Ratio Analysis
if raw_annotations:
    widths = [a["bbox"][2] for a in raw_annotations if len(a.get("bbox", [])) == 4 and a["bbox"][2] > 0]
    heights = [a["bbox"][3] for a in raw_annotations if len(a.get("bbox", [])) == 4 and a["bbox"][3] > 0]
    aspect_ratios = [w / max(1.0, h) for w, h in zip(widths, heights)]
    
    fig, axs = plt.subplots(1, 3, figsize=(15, 4))
    axs[0].hist(widths, bins=30, color="#0091EA", edgecolor="black")
    axs[0].set_title("Box Width (px)")
    axs[0].set_xlabel("Width")
    
    axs[1].hist(heights, bins=30, color="#00C853", edgecolor="black")
    axs[1].set_title("Box Height (px)")
    axs[1].set_xlabel("Height")
    
    axs[2].hist(aspect_ratios, bins=30, color="#FF9100", edgecolor="black")
    axs[2].set_title("Aspect Ratio (Width / Height)")
    axs[2].set_xlabel("Ratio")
    
    plt.tight_layout()
    plt.show()
    
    print(f"Median Box Width:  {np.median(widths):.1f} px")
    print(f"Median Box Height: {np.median(heights):.1f} px")
    print(f"Median Aspect:     {np.median(aspect_ratios):.2f}")
else:
    print("No bounding boxes to plot.")

In [ ]:
# CELL 6: Tag Density per Image
if raw_images and raw_annotations:
    anns_per_img = Counter([a["image_id"] for a in raw_annotations])
    counts = [anns_per_img.get(img["id"], 0) for img in raw_images]
    
    print("=== Tag Density per Image ===")
    print(f"Min tags per image:  {min(counts)}")
    print(f"Max tags per image:  {max(counts)}")
    print(f"Mean tags per image: {np.mean(counts):.2f}")
    print(f"Images with 0 tags:  {counts.count(0)}")
    
    plt.figure(figsize=(7, 3.5))
    plt.hist(counts, bins=min(25, max(1, max(counts))), color="#5856D6", edgecolor="black")
    plt.title("Distribution of Tags per Image", fontsize=12)
    plt.xlabel("Number of Tags")
    plt.ylabel("Number of Images")
    plt.tight_layout()
    plt.show()

In [ ]:
# CELL 7: EDA Summary & Next Steps
print("""
==============================================================================
 EDA SUMMARY & WORKFLOW RECOMMENDATIONS:
 1. Category balance and bounding box distributions have been profiled.
 2. Proceed to folder '02_anomalies/detect_and_audit_anomalies.ipynb'
    to audit labeling quality, flag corrupted annotations, and export
    the clean dataset splits (excluding anomalies) for training.
==============================================================================
""")